# Pipeline walkthrough

This notebook runs the same cleaning pipeline as `clean.py`, **one step at a time**, so you can see what each step does to the data.

The logic lives in `cleaning/*.py`. The notebook only calls it and shows the results, so the script and the notebook can never disagree.

Run `python scripts/generate_messy_data.py --out data/raw` once before this notebook.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# Find the project folder (the one that contains `cleaning/`), wherever Jupyter started
ROOT = Path.cwd()
while not (ROOT / "cleaning").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "cleaning").is_dir():
    raise FileNotFoundError(f"Can't find the project from {Path.cwd()}. "
                            "Put this notebook inside the project's notebooks/ folder.")
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

from cleaning import normalize as nz
from cleaning import pipeline

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

Project folder: /home/claude/sales-data-cleanup


## Step 1: Load every export as text

Everything is read as text (`dtype=str`) so phone numbers keep their leading 0. Column names from the 3 systems are mapped to one schema.

In [2]:
raw, file_info = pipeline.load_exports(ROOT / "data" / "raw")
print(f"{len(raw):,} rows loaded")
pd.DataFrame(file_info)

4,698 rows loaded


,file,rows,columns
0,orders_2026_01.csv,713,11
1,orders_2026_02.csv,847,11
2,orders_2026_03.xlsx,777,11
3,orders_2026_04.xlsx,792,11
4,orders_2026_05.csv,754,11
5,orders_2026_06.csv,815,11


In [3]:
# Same column, three different names in the original files
for f in sorted((ROOT / "data" / "raw").iterdir()):
    df = pd.read_excel(f, nrows=0) if f.suffix == ".xlsx" else pd.read_csv(f, nrows=0)
    print(f"{f.name:22} {list(df.columns)[:4]}")

orders_2026_01.csv     ['Order ID', 'Order Date', 'Customer Name', 'Phone']
orders_2026_02.csv     ['Order ID', 'Order Date', 'Customer Name', 'Phone']
orders_2026_03.xlsx    ['order_no', 'date', 'customer', 'mobile']
orders_2026_04.xlsx    ['order_no', 'date', 'customer', 'mobile']
orders_2026_05.csv     ['OrderID', 'Date', 'Client', 'Phone Number']
orders_2026_06.csv     ['OrderID', 'Date', 'Client', 'Phone Number']


In [4]:
raw.sample(5, random_state=1)

,order_id,order_date,customer_name,phone,email,city,product,quantity,unit_price,payment_method,status,source_file,file_order,source_row
3291,ORD 13258,"May 20, 2026",Khaled Ramadan,01090232106,,cairo,Memory Foam Pillow,1,750.0,Vodafone Cash,delivered,orders_2026_05.csv,4,164
4221,13748,"Jun 17, 2026",Omar Mansour,010 4166 2236,omar.mansour60@yahoo.com,CAIRO,Ceramic Dinner Set (24 pcs),1,2300.0,COD,DELIVERED,orders_2026_06.csv,5,340
2872,12168,20/04/2026,Kirollos Ezzat,1557405358,kirollos.ezzat66@yahoo.com,Cairo City,Ceramic Dinner Set (24 pcs),1,"2,300.00 EGP",COD,Delivered,orders_2026_04.xlsx,3,537
4041,ORD-13431,"Jun 11, 2026",Marina Gamal,010 9264 7902,marina.gamal48@gmail.com,Mansoura,Non-stick Frying Pan 28cm,2,840.0,Cash on Delivery,Cancelled,orders_2026_06.csv,5,160
3591,12880,"May 20, 2026",HABIBA KAMAL,1562336371,HABIBA.KAMAL27@YAHOO.COM,tanta,Cotton Bed Sheet Set,1,1400.0,cash,Completed,orders_2026_05.csv,4,464


## Step 2: Drop fully empty rows

In [5]:
raw, empty_rows = pipeline.drop_empty_rows(raw)
print(f"dropped {empty_rows} empty rows, {len(raw):,} left")

dropped 43 empty rows, 4,655 left


## Step 3: Clean one value at a time

Every column has a small function in `cleaning/normalize.py`. Try them on single values first.

In [6]:
phones = ["01012345678", "00201012345678", "010 1234 5678", "+20 101-234-5678", "1012345678", "12345", "N/A"]
pd.DataFrame({"raw": phones, "clean": [nz.phone(p) for p in phones]})

,raw,clean
0,01012345678,+201012345678
1,00201012345678,+201012345678
2,010 1234 5678,+201012345678
3,+20 101-234-5678,+201012345678
4,1012345678,+201012345678
5,12345,NaN
6,N/A,NaN


In [7]:
samples = {
    "order_id": ["ord10001", "ORD 10001", "10001"],
    "order_date": ["2026-03-09", "09/03/2026", "Mar 09, 2026"],
    "customer_name": ["  mostafa   ADEL ", "MOSTAFA ADEL", "Mostafa Adel"],
    "city": ["Al Qahirah", " CAIRO ", "Cairo City"],
    "quantity": ["2 pcs", "0", "-2"],
    "unit_price": ["2,300.00 EGP", "EGP 2,400", ""],
}
rows = []
for col, values in samples.items():
    fn = pipeline.CLEANERS[col]
    rows += [{"column": col, "raw": v, "clean": fn(v)} for v in values]
pd.DataFrame(rows)

,column,raw,clean
0,order_id,ord10001,ORD-10001
1,order_id,ORD 10001,ORD-10001
2,order_id,10001,ORD-10001
3,order_date,2026-03-09,2026-03-09
4,order_date,09/03/2026,2026-03-09
5,order_date,"Mar 09, 2026",2026-03-09
6,customer_name,mostafa ADEL,Mostafa Adel
7,customer_name,MOSTAFA ADEL,Mostafa Adel
8,customer_name,Mostafa Adel,Mostafa Adel
9,city,Al Qahirah,Cairo


Now apply every function to every row. `changes` counts, per column, how many values were standardized, unusable or missing.

In [8]:
clean, changes = pipeline.standardize(raw)
pd.DataFrame(changes).T

,standardized,unusable,missing
order_id,3514,0,0
order_date,3110,0,0
customer_name,1893,0,0
phone,3771,34,75
email,551,137,253
city,3427,0,0
product,0,0,0
quantity,475,96,0
unit_price,1998,0,43
payment_method,4655,0,0


In [9]:
# The same 5 rows, before and after
cols = ["order_id", "order_date", "customer_name", "phone", "city", "quantity", "unit_price", "status"]
idx = raw.sample(5, random_state=7).index
pd.concat({"before": raw.loc[idx, cols], "after": clean.loc[idx, cols]}, axis=1)

before                                                                                           after                                                                                  
       order_id    order_date customer_name           phone      city quantity unit_price     status   order_id  order_date customer_name          phone      city quantity unit_price     status
3283  ORD-12766  May 15, 2026     omar nasr   010 1149 6211    CAIRO         3      650.0  Delivered  ORD-12766  2026-05-15     Omar Nasr  +201011496211     Cairo        3      650.0  delivered
567   ORD 10170    2026-01-12  HANA SOLIMAN  00201556240084     TANTA    1 pcs      420.0  DELIVERED  ORD-10170  2026-01-12  Hana Soliman  +201556240084     Tanta        1      420.0  delivered
4696  ORD-13785  Jun 23, 2026  Karim Farouk   010 6771 9420     Cairo        2    EGP 890  delivered  ORD-13785  2026-06-23  Karim Farouk  +201067719420     Cairo        2      890.0  delivered
3501      12768  May 06, 2026  Yasmin Fayez                  mansoura        1      730.0  Completed  ORD-12768  2026-05-06  Yasmin Fayez            NaN  Mansoura        1      730.0  delivered
196       10612    2026-01-16  Fady Mansour   +201297627170   El Giza        1      490.0  Completed  ORD-10612  2026-01-16  Fady Mansour  +201297627170      Giza        1      490.0  delivered

## Step 4: Reject rows missing a required field

Nothing is guessed. A row without a usable quantity or price goes to the rejected log with its reason.

In [10]:
valid, rejected = pipeline.split_valid(clean, raw)
print(f"{len(valid):,} valid rows, {len(rejected):,} rejected")
rejected["reject_reason"].value_counts()

4,519 valid rows, 136 rejected


reject_reason
invalid or missing quantity                                   93
invalid or missing unit_price                                 40
invalid or missing quantity; invalid or missing unit_price     3
Name: count, dtype: int64

In [11]:
rejected[["source_file", "source_row", "order_id", "quantity", "unit_price", "reject_reason"]].head()

,source_file,source_row,order_id,quantity,unit_price,reject_reason
9,orders_2026_01.csv,11,10233,-2,2400.0,invalid or missing quantity
49,orders_2026_01.csv,51,ord10464,0,365.0,invalid or missing quantity
53,orders_2026_01.csv,55,ORD-10305,1,,invalid or missing unit_price
64,orders_2026_01.csv,66,ord10417,-3,1450.0,invalid or missing quantity
96,orders_2026_01.csv,98,ord10532,0,360.0,invalid or missing quantity


## Step 5: Remove duplicates

First repeats inside the same file, then the same order exported again in a later file. The copy from the **latest file** is kept, because it has the newest status.

In [12]:
final, dupes = pipeline.deduplicate(valid)
print(dupes)
print(f"{len(final):,} unique orders")

{'within_file': 130, 'across_files': 534, 'conflicting_orders': 28}
3,855 unique orders


In [13]:
# One order whose status changed between exports
changed = valid.groupby("order_id")["status"].nunique()
oid = changed[changed > 1].index[0]
print("raw copies:")
display(valid.loc[valid["order_id"] == oid, ["order_id", "source_file", "status"]])
print("kept:")
final.loc[final["order_id"] == oid, ["order_id", "source_file", "status"]]

raw copies:


,order_id,source_file,status
317,ORD-10101,orders_2026_01.csv,pending
948,ORD-10101,orders_2026_02.csv,delivered


kept:


,order_id,source_file,status
948,ORD-10101,orders_2026_02.csv,delivered


## Step 6: Build the outputs

Customers are grouped by phone number, because the same person's name is spelled several ways.

In [14]:
orders = pipeline.build_orders(final)
customers, merged = pipeline.build_customers(orders)
print(f"{len(orders):,} orders, {len(customers):,} customers, {merged} customers had more than one name spelling")
orders.head()

3,855 orders, 350 customers, 66 customers had more than one name spelling


,order_id,order_date,customer_name,phone,email,city,product,quantity,unit_price,line_total,payment_method,status,source_file
0,ORD-10079,2026-01-01,Karim Ramadan,+201163280554,karim.ramadan4@outlook.com,Tanta,Memory Foam Pillow,2,650.0,1300.0,card,pending,orders_2026_01.csv
1,ORD-10126,2026-01-01,Habiba Kamal,+201562336371,habiba.kamal27@yahoo.com,Tanta,Memory Foam Pillow,1,650.0,650.0,mobile_wallet,delivered,orders_2026_01.csv
2,ORD-10159,2026-01-01,Nada Mansour,+201555177213,NaN,Cairo,Memory Foam Pillow,1,650.0,650.0,cash_on_delivery,cancelled,orders_2026_01.csv
3,ORD-10211,2026-01-01,Fatma Mahmoud,+201510106509,fatma.mahmoud19@gmail.com,Mansoura,Bath Towel Pack (4),2,780.0,1560.0,card,delivered,orders_2026_01.csv
4,ORD-10266,2026-01-01,Yasmin Fathy,+201085726382,yasmin.fathy38@yahoo.com,Tanta,Bath Towel Pack (4),1,880.0,880.0,cash_on_delivery,delivered,orders_2026_01.csv


In [15]:
# One customer, several spellings, one phone
phone = customers.loc[customers["name_spellings"] > 1, "phone"].iloc[0]
raw_names = raw.loc[clean["phone"] == phone, "customer_name"].unique()
print("spellings in the raw files:", list(raw_names))
customers[customers["phone"] == phone]

spellings in the raw files: ['Amr Kamal', 'AMRO KAMAL', '  Amr   Kamal ', 'amr kamal', '  Amro   Kamal ', 'AMR KAMAL', 'Amro Kamal']


,phone,customer_name,name_spellings,email,city,orders,delivered_revenue_egp,first_order,last_order
4,+201110043277,Amr Kamal,2,amr.kamal61@yahoo.com,Alexandria,12,26535.0,2026-01-03,2026-06-21


## Check: every row is accounted for

In [16]:
flow = {
    "raw rows": len(raw) + empty_rows,
    "empty": -empty_rows,
    "rejected": -len(rejected),
    "duplicates inside a file": -dupes["within_file"],
    "re-exported in a later file": -dupes["across_files"],
}
flow["clean orders"] = sum(flow.values())
assert flow["clean orders"] == len(orders)
pd.Series(flow, name="rows").to_frame()

,rows
raw rows,4698
empty,-43
rejected,-136
duplicates inside a file,-130
re-exported in a later file,-534
clean orders,3855
